In [ ]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re
from datetime import datetime


nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

In [ ]:

db_books_file = Path(project_root / "data_reload/db_files/books.json")
db_b2p_file = Path(project_root / "data_reload/db_files/books2people.json")
db_admin_file = Path(project_root / "data_reload/db_files/book_admin.json")
db_people_file = Path(project_root / "data_reload/db_files/people.json")

with open(db_books_file, "r") as f:
   books = json.load(f)
with open(db_b2p_file, "r") as f:
   b2p = json.load(f)
with open(db_admin_file, "r") as f:
   admin = json.load(f)
with open(db_people_file, "r") as f:
   people = json.load(f)


books_dict = {i["composite_id"]: i for i in books}
b2p_dict = {i["composite_id"]: i for i in b2p}
people_dict = {i["unified_id"]: i for i in people}
rprint(dict(list(people_dict.items())[:2]))


missing_dict = {k: v for k, v in books_dict.items() if k not in b2p_dict}
# with open("missing_b2p_entries_initial_version.json", "w") as f:
#     json.dump(missing_dict, f, ensure_ascii=False, indent=2)
remaining_cids = set(missing_dict.keys())
with open("remaining_cids.json", "w") as f:
    json.dump(list(remaining_cids), f)
rprint(dict(list(missing_dict.items())[:1]))
rprint(f"missing rows: {len(missing_dict)}")

I now have all the raw book table data but not the admin_data to check for details. 

In @find_missing_b2p.ipynb I first checked for a hunch that didn't pan out. 

Then in cells 4 & 5 I built different versions of my data to check for existing people etc. 

I then went searching for composite_ids in my backup data and found that people_matched.json contains 1337 composite_ids with people & roles. 

so instead of building on already somewhat messily transformed data, I'll go ahead and extract that number of rows from that file directly, build the b2p list file for the insert/update and then look at the rest. 

**end of notebook: save what's left**
```
with open("remaining_cids.json", "w") as f:
    json.dump(list(remaining_cids), f)
```

**start of next notebook: load it back**
```
with open("remaining_cids.json") as f:
    remaining_cids = set(json.load(f))
```


In [ ]:
matched_people_file = Path(project_root / "data/people/prepping4loading/people_matched.json")

with open(matched_people_file, "r") as f:
   matched_people = json.load(f)

match_uid_dict = {v["unified_id"]: {
   "entries": v["occurrences"]
} for k, v in matched_people.items()}
# rprint(dict(list(matched_people.items())[:2]))


match_cid_dict = {}
for person in matched_people.values():
    uid = person["unified_id"]
    for occ in person["occurrences"]:
        cid = occ["composite_id"]
        match_cid_dict.setdefault(cid, []).append({
            "book_id": None,
            "composite_id": cid,
            "person_id": None,
            "unified_id": uid,
            "display_name": occ["display_name"],
            "family_name": None,
            "given_names": None,
            "name_prefix": None,
            "name_particles": None,
            "name_suffix": None,
            "single_name": None,
            **occ})

# rprint(dict(list(match_cid_dict.items())[:2]))

comp_id_in_matched = {}
not_found = set()
found = set()



for cid, entries in match_cid_dict.items():
    if cid in missing_dict:
        found.add(cid)
        for entry in entries:
            uid = entry["unified_id"]
            book_id = books_dict[cid]["book_id"]
            person_id = people_dict[uid]["person_id"]
            comp_id_in_matched.setdefault(cid,[]).append({
                **entry,
                "book_id": book_id,
                "person_id": person_id
            })


# THIS OVERWRITES THE LIST ENTRIES!
            # comp_id_in_matched[cid] = {
            #     **entry,
            #     "book_id": book_id,
            #     "person_id": person_id
            # }
rprint(dict(list(comp_id_in_matched.items())[:1]))

# I need to unwrap the inner lists first
found_list = [v for v in comp_id_in_matched.values() for v in v]


with open("b2p_found_list_01.json", "w") as f:
    json.dump(found_list, f, ensure_ascii=False, indent=2)

remaining_cids = remaining_cids - found
rprint(f"remaining: {len(remaining_cids)}")


with open("remaining_cids.json", "w") as f:
    json.dump(list(remaining_cids), f)

# with open("b2p_found_data_01.json", "w") as f:
#     json.dump(comp_id_in_matched, f, ensure_ascii=False, indent=2)
